# 02 · Would the model code it the same way twice?

> **Old question:** Is our coding reliable? Would a second coder agree with the first?
> **New tool:** the same LLM as two "coders": one run at temperature 0 and one at temperature 0.7.
> **What you'll see:** an agreement rate, and the tasks where the two runs disagree. These are where coding is genuinely ambiguous.

## Background

We classify **O\*NET work tasks** by how *exposed* they are to large language models. O\*NET is the US Department of Labor's database describing what people do in about 1,000 occupations. Economists use task-level exposure scores like these to study which jobs AI might change.

**Temperature** controls randomness. At 0 the model (almost) always picks its most likely answer; at 0.7 it sometimes picks a less likely one. If a task's label changes between runs, the model was not sure, and a human coder may not be either.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from typing import Literal
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field
from workshop import *

setup_plots()
status()

In [ ]:
tasks = pd.read_csv("../data/onet_tasks.csv")
gold = pd.read_csv("../data/gold_onet.csv")
sample = tasks.merge(gold[["task_id"]], on="task_id")   # the 40 tasks we also use in notebook 03
print(len(sample), "tasks from", sample.occupation_title.nunique(), "occupations")
sample[["occupation_title", "task"]].head()

## The rubric

A short rubric with three levels, written as the instructions plus a codebook. A `reason` field makes each answer easier to inspect.

In [ ]:
RUBRIC = """You code O*NET work tasks for exposure to large language models (LLMs) such as ChatGPT.
high: an LLM alone could do most of this task today, because it is mainly reading, writing or analysing text or data.
medium: an LLM could do part of the task, or could do it only with extra tools, data access or human checks.
low: the task is mainly physical or in-person, or depends on presence, relationships or formal authority."""

class Exposure(BaseModel):
    label: Literal["high", "medium", "low"]
    reason: str = Field(description="One short sentence explaining the label")

## Two runs

In [ ]:
run_a = extract_many(sample, "task", Exposure, RUBRIC, temperature=0.0)
run_b = extract_many(sample, "task", Exposure, RUBRIC, temperature=0.7)
both = sample.assign(label_t0=run_a["label"], label_t07=run_b["label"],
                     reason_t0=run_a["reason"], reason_t07=run_b["reason"])
print("failed rows:", run_a["error"].notna().sum() + run_b["error"].notna().sum())
both = both.dropna(subset=["label_t0", "label_t07"])   # keep tasks coded in both runs

In [ ]:
agree = (both["label_t0"] == both["label_t07"]).mean()
print(f"Agreement between the two runs: {agree:.0%} of {len(both)} tasks")
plot_confusion(both["label_t0"], both["label_t07"], ["high", "medium", "low"],
               "temperature 0", "temperature 0.7", "Do two runs give a task the same label?");

This is a **confusion matrix**. Each row is a label from the first run (temperature 0); each column is a label from the second run (temperature 0.7). The number in a cell counts the tasks that got that pair of labels.

- The **outlined diagonal** is agreement: same label both times.
- **Off-diagonal cells** show *which* labels get confused. Usually it's high vs medium or medium vs low, rarely high vs low. Disagreement between neighbouring levels is a sign the boundary in the rubric is fuzzy.

In Session 2 we use exactly this picture to compare the model with human coders.

In [ ]:
order = ["high", "medium", "low"]
counts = pd.DataFrame({"temperature 0": both["label_t0"].value_counts(),
                       "temperature 0.7": both["label_t07"].value_counts()}).reindex(order).fillna(0)
ax = counts.plot.bar(rot=0, width=0.7, edgecolor="#fcfcfb", linewidth=2)
ax.set_title("Same rubric, same tasks: label counts in two runs")
ax.set_ylabel("number of tasks"); ax.set_xlabel("")
plt.show()

## Where do the runs disagree?

These are the interesting rows. Read the task and both reasons: is the task really ambiguous, or is the rubric unclear?

In [ ]:
disagree = both[both["label_t0"] != both["label_t07"]]
print(len(disagree), "tasks with different labels\n")
for r in disagree.itertuples():
    print(f"[{r.occupation_title}] {r.task}")
    print(f"   temperature 0   → {r.label_t0:<6}  because: {r.reason_t0}")
    print(f"   temperature 0.7 → {r.label_t07:<6}  because: {r.reason_t07}\n")

**Discussion:** agreement between two runs of one model is the *floor*, not proof of validity. A model can be consistently wrong. Notebook 03 and Session 2 compare the labels with human coding.

In [ ]:
pathlib.Path("../outputs").mkdir(exist_ok=True)
both.to_csv("../outputs/onet_exposure_runs.csv", index=False)
print("saved outputs/onet_exposure_runs.csv (used in notebook 03)")

## Try it yourself

Edit the rubric: for example, add an example task to each level, or merge `medium` into the other two. Re-run the two runs and see whether agreement goes up. A clearer rubric usually means fewer disagreements, for humans and models alike.

In [ ]:
# Your turn: change RUBRIC above, re-run it, then run this cell on 10 tasks
test = extract_many(sample.head(10), "task", Exposure, RUBRIC, temperature=0.7)
test[["task", "label", "reason"]]